# Projeto Integrador: Fábrica Inteligente - MetalTech Soluções Industriais S.A.
## Etapa 3: Visão Computacional para Inspeção Óptica de Qualidade

Este notebook implementa a esteira completa de visão computacional em **OpenCV**, aplicando:
1. Filtros espaciais (Escala de cinza e suavização Gaussiana 5x5 para atenuação de ruídos)
2. Detecção de bordas por histerese com operador **Canny (50/150)**
3. Extração e parametrização geométrica de contornos (Área nominal e Circularidade ISO)
4. Detecção espectral de oxidação/ferrugem no espaço de cor **HSV**
5. Classificação automatizada entre peças aprovadas e defeituosas (trincadas/corroídas).


In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from visao_computacional import InspetorQualidadeVisual

# Configurações estéticas de gráficos
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
print('Módulos de visão computacional importados com sucesso!')


### 1. Carregamento e Inspeção Inicial da Peça Padrão
Carregamento da imagem gerada pelo sensor óptico da linha de usinagem.

In [ ]:
caminho_peca = Path('images/peca_teste.jpg')
imagem_bgr = cv2.imread(str(caminho_peca))
if imagem_bgr is None:
    raise FileNotFoundError(f'Imagem não localizada: {caminho_peca}')

imagem_rgb = cv2.cvtColor(imagem_bgr, cv2.COLOR_BGR2RGB)
print('Dimensões da imagem (H, W, C):', imagem_bgr.shape)

plt.figure(figsize=(7, 5))
plt.imshow(imagem_rgb)
plt.title('Peça em Inspeção - Imagem Original (RGB)')
plt.axis('off')
plt.show()


### 2. Filtros de Pré-Processamento: Escala de Cinza e Gaussian Blur
Eliminação de ruídos de alta frequência preservando o contraste estrutural da peça.

In [ ]:
imagem_cinza = cv2.cvtColor(imagem_bgr, cv2.COLOR_BGR2GRAY)
imagem_blur = cv2.GaussianBlur(imagem_cinza, (5, 5), 0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(imagem_rgb)
axes[0].set_title('1. Original (RGB)')
axes[0].axis('off')

axes[1].imshow(imagem_cinza, cmap='gray')
axes[1].set_title('2. Escala de Cinza')
axes[1].axis('off')

axes[2].imshow(imagem_blur, cmap='gray')
axes[2].set_title('3. Suavização Gaussiana (Filtro Passa-Baixas)')
axes[2].axis('off')

plt.tight_layout()
plt.show()


### 3. Detecção de Bordas Canny (Duplo Limiar de Histerese 50/150)

In [ ]:
bordas = cv2.Canny(imagem_blur, threshold1=50, threshold2=150)

plt.figure(figsize=(7, 5))
plt.imshow(bordas, cmap='gray')
plt.title('Detecção de Bordas por Gradiente Canny')
plt.axis('off')
plt.show()


### 4. Extração, Parametrização e Análise Geométrica de Contornos

In [ ]:
contornos, hierarquia = cv2.findContours(bordas.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contornos_validos = [c for c in contornos if cv2.contourArea(c) >= 50]

imagem_contornos = imagem_rgb.copy()
cv2.drawContours(imagem_contornos, contornos_validos, -1, (0, 255, 0), 2)

dados_geometricos = []
for idx, c in enumerate(contornos_validos, 1):
    area = cv2.contourArea(c)
    peri = cv2.arcLength(c, True)
    circ = (4 * np.pi * area) / (peri**2) if peri > 0 else 0.0
    x, y, w, h = cv2.boundingRect(c)
    dados_geometricos.append({
        'Contorno': idx,
        'Área (px²)': round(area, 2),
        'Perímetro (px)': round(peri, 2),
        'Circularidade': round(circ, 3),
        'Largura': w,
        'Altura': h
    })

df_geo = pd.DataFrame(dados_geometricos)
print(f'Total de contornos externos identificados: {len(contornos_validos)}')
display(df_geo)

plt.figure(figsize=(7, 5))
plt.imshow(imagem_contornos)
plt.title('Contornos Externos Reconhecidos (Verde)')
plt.axis('off')
plt.show()


### 5. Esteira Automatizada de Classificação Industrial (Aprovado vs Reprovado)
Avaliação em lote do conjunto de amostras: Peça Nominal, Peça com Trinca Mecânica e Peça com Oxidação/Ferrugem.

In [ ]:
inspetor = InspetorQualidadeVisual()
pasta_imgs = Path('images')
amostras = sorted([f for f in pasta_imgs.glob('*.jpg') if 'inspecoes' not in str(f)])

resultados_lote = []
fig, axes = plt.subplots(len(amostras), 2, figsize=(12, 3.5 * len(amostras)))

for idx, caminho in enumerate(amostras):
    res, anotada = inspetor.inspecionar(caminho)
    resultados_lote.append({
        'Arquivo': res.arquivo,
        'Status': res.status,
        'Área (px²)': res.area_principal,
        'Circularidade': res.circularidade,
        'Motivo / Diagnóstico': res.motivo
    })
    
    orig = cv2.cvtColor(cv2.imread(str(caminho)), cv2.COLOR_BGR2RGB)
    anotada_rgb = cv2.cvtColor(anotada, cv2.COLOR_BGR2RGB)
    axes[idx, 0].imshow(orig)
    axes[idx, 0].set_title(f'Entrada: {res.arquivo}')
    axes[idx, 0].axis('off')
    axes[idx, 1].imshow(anotada_rgb)
    axes[idx, 1].set_title(f'Resultado: {res.status}')
    axes[idx, 1].axis('off')

plt.tight_layout()
plt.show()

df_resultado = pd.DataFrame(resultados_lote)
display(df_resultado)
